# Generación de archivo BibTeX desde referencias_unificado.csv

Este notebook genera un archivo .bib con todas las referencias usando sus DOIs para facilitar la importación a Mendeley.

In [1]:
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm
import re

## 1. Cargar el CSV con las referencias ordenadas

In [2]:
# Cargar el CSV con las referencias ordenadas
df = pd.read_csv('./referencias_unificado.csv', sep='|', encoding='utf-8')

print(f"Total de referencias cargadas: {len(df)}")
print(f"Referencias con DOI: {df['DOI'].notna().sum()}")
print(f"\nPrimeras 5 referencias:")
print(df[['title', 'DOI', 'year']].head())

Total de referencias cargadas: 135
Referencias con DOI: 135

Primeras 5 referencias:
                                               title  \
0  Multi-Class Classification of Plant Leaf Disea...   
1  YOLO-RDM: Innovative Detection Methods for Egg...   
2  Constructing and Optimizing RNN Models to Pred...   
3  A Review on Automated Detection and Assessment...   
4  IMNM: integrated multi-network model for ident...   

                           DOI  year  
0  10.1109/ACCESS.2023.3286730  2023  
1  10.1109/ACCESS.2025.3545670  2025  
2  10.1109/ACCESS.2023.3311477  2023  
3  10.1109/ACCESS.2024.3362230  2024  
4    10.3389/fpls.2025.1558349  2025  


## 2. Función para obtener BibTeX desde DOI usando CrossRef

In [3]:
def get_bibtex_from_doi(doi, timeout=10):
    """
    Obtiene la entrada BibTeX desde un DOI usando la API de CrossRef.
    
    Args:
        doi (str): El DOI de la referencia
        timeout (int): Tiempo máximo de espera para la petición
    
    Returns:
        str: Entrada BibTeX o None si falla
    """
    if pd.isna(doi):
        return None
    
    try:
        # CrossRef Content Negotiation - pedir formato BibTeX
        url = f"https://doi.org/{doi}"
        headers = {
            'Accept': 'application/x-bibtex',
            'User-Agent': 'Mozilla/5.0 (Python script for academic research)'
        }
        
        response = requests.get(url, headers=headers, timeout=timeout)
        
        if response.status_code == 200:
            bibtex_entry = response.text.strip()
            return bibtex_entry
        else:
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Error obteniendo BibTeX para DOI {doi}: {e}")
        return None

# Probar con el primer DOI
test_doi = df[df['DOI'].notna()]['DOI'].iloc[0]
print(f"Probando con DOI: {test_doi}\n")
test_bibtex = get_bibtex_from_doi(test_doi)
if test_bibtex:
    print("✓ Función funcionando correctamente")
    print("\nEjemplo de entrada BibTeX:")
    print(test_bibtex[:500] + "..." if len(test_bibtex) > 500 else test_bibtex)
else:
    print("✗ No se pudo obtener BibTeX")

Probando con DOI: 10.1109/ACCESS.2023.3286730

✓ Función funcionando correctamente

Ejemplo de entrada BibTeX:
@article{Hosny_2023, title={Multi-Class Classification of Plant Leaf Diseases Using Feature Fusion of Deep Convolutional Neural Network and Local Binary Pattern}, volume={11}, ISSN={2169-3536}, url={http://dx.doi.org/10.1109/ACCESS.2023.3286730}, DOI={10.1109/access.2023.3286730}, journal={IEEE Access}, publisher={Institute of Electrical and Electronics Engineers (IEEE)}, author={Hosny, Khalid M. and El-Hady, Walaa M. and Samy, Farid M. and Vrochidou, Eleni and Papakostas, George A.}, year={2023...


## 3. Generar entradas BibTeX para todas las referencias

In [4]:
# Filtrar solo referencias con DOI
df_with_doi = df[df['DOI'].notna()].copy()

print(f"Referencias con DOI para procesar: {len(df_with_doi)}")
print("\n⚠️ Este proceso puede tardar varios minutos...")
print("Obteniendo entradas BibTeX desde CrossRef...\n")

# Lista para almacenar las entradas BibTeX
bibtex_entries = []
failed_dois = []

# Procesar cada DOI con barra de progreso
for idx, row in tqdm(df_with_doi.iterrows(), total=len(df_with_doi), desc="Obteniendo BibTeX"):
    doi = row['DOI']
    bibtex = get_bibtex_from_doi(doi)
    
    if bibtex:
        bibtex_entries.append(bibtex)
    else:
        failed_dois.append(doi)
    
    # Pausa breve para no saturar la API
    time.sleep(0.3)

print(f"\n✓ Proceso completado")
print(f"  - Entradas BibTeX obtenidas: {len(bibtex_entries)}")
print(f"  - DOIs fallidos: {len(failed_dois)}")

if failed_dois:
    print(f"\n⚠️ Los siguientes DOIs no pudieron ser procesados:")
    for doi in failed_dois[:10]:  # Mostrar solo los primeros 10
        print(f"  - {doi}")
    if len(failed_dois) > 10:
        print(f"  ... y {len(failed_dois) - 10} más")

Referencias con DOI para procesar: 135

⚠️ Este proceso puede tardar varios minutos...
Obteniendo entradas BibTeX desde CrossRef...



Obteniendo BibTeX:   0%|          | 0/135 [00:00<?, ?it/s]


✓ Proceso completado
  - Entradas BibTeX obtenidas: 135
  - DOIs fallidos: 0


## 4. Guardar todas las entradas en un archivo .bib

In [5]:
# Nombre del archivo de salida
output_bib = './referencias_completas.bib'

# Escribir todas las entradas BibTeX en el archivo
with open(output_bib, 'w', encoding='utf-8') as f:
    for entry in bibtex_entries:
        f.write(entry)
        f.write('\n\n')  # Doble salto de línea entre entradas

print(f"✓ Archivo BibTeX generado exitosamente: {output_bib}")
print(f"  Total de entradas: {len(bibtex_entries)}")
print(f"\n📝 Ahora puedes:")
print(f"  1. Abrir Mendeley")
print(f"  2. Ir a File > Import > {output_bib}")
print(f"  3. Mendeley descargará automáticamente los PDFs disponibles usando los DOIs")

✓ Archivo BibTeX generado exitosamente: ./referencias_completas.bib
  Total de entradas: 135

📝 Ahora puedes:
  1. Abrir Mendeley
  2. Ir a File > Import > ./referencias_completas.bib
  3. Mendeley descargará automáticamente los PDFs disponibles usando los DOIs


## 5. (Opcional) Guardar lista de DOIs fallidos para revisión manual

In [6]:
if failed_dois:
    # Guardar DOIs fallidos en un archivo de texto
    failed_file = './dois_fallidos.txt'
    with open(failed_file, 'w', encoding='utf-8') as f:
        f.write("DOIs que no pudieron ser procesados:\n\n")
        for doi in failed_dois:
            f.write(f"{doi}\n")
    
    print(f"✓ Lista de DOIs fallidos guardada en: {failed_file}")
    print(f"  Puedes intentar buscarlos manualmente o usar otro método")
else:
    print("✓ Todos los DOIs fueron procesados exitosamente")

✓ Todos los DOIs fueron procesados exitosamente
